In [3]:
from anthropic import Anthropic
client = Anthropic()

In [4]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [5]:
messages=[
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = client.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=1024,
    messages=messages
)

response.content[0].text

"I'd be happy to help you join a course! However, I need a bit more information since I don't have context about which course you're referring to. Could you tell me:\n\n1. **Which course** are you interested in?\n2. **Where did you find it** (a specific platform, school, or organization)?\n3. **Any enrollment details** you've already seen (dates, requirements, etc.)?\n\nOnce you share these details, I can give you more specific guidance on how to enroll!"

In [6]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [7]:
search_tool = {
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "input_schema": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"]
    }
}

In [8]:
response = client.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=1024,
    messages=messages,
    tools=[search_tool]
)

In [9]:
len(response.content)

2

In [10]:
response.content[1]

ToolUseBlock(id='toolu_01AfU7ftHzbi9VW2mexPZ4ts', caller=DirectCaller(type='direct'), input={'query': 'how to join course registration enrollment'}, name='search', type='tool_use')

In [11]:
tool_use = response.content[1]

In [12]:
tool_use

ToolUseBlock(id='toolu_01AfU7ftHzbi9VW2mexPZ4ts', caller=DirectCaller(type='direct'), input={'query': 'how to join course registration enrollment'}, name='search', type='tool_use')

In [13]:
args = tool_use.input
print(args)

{'query': 'how to join course registration enrollment'}


In [14]:
results = search(args['query'])
print(results)

[{'id': '74eb249bbf', 'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'I just discovered the course. Can I still join?', 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}, {'id': '04919992b3', 'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'How should I start the course and follow the weekly workflow?', 'answer': 'Start with the [LLM Zoomcamp docs](https://datatalks.club/docs/courses/llm-zoomcamp/), the [general Zoomcamp logistics docs](https://datatalks.club/docs/courses/zoomcamp-logistics/), and the [LLM Zoomcamp GitHub repository](https://github.com/DataTalksClub/llm-zoomcamp).\n\nYou can start whenever you want. The videos and GitHub materials are available, and the deadlines are listed in the [course management platform](https://courses.datatalks.club/llm-zoomcamp-2026/).\n\nA typical workflow is:\n\n1. Watch the lesson video

What we've done so far:

1. Make a call to the LLM <-- (first call)
2. LL decided to invoke search with 'params'
3. Then we invoked the search and got results back
4. Now we'll send the results back to the LLM <-- (another call)

In [15]:
#turning results into json
import json

results_json = json.dumps(results, indent = 2)
print(results_json)

[
  {
    "id": "74eb249bbf",
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "I just discovered the course. Can I still join?",
    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\u2019re still accepting submissions."
  },
  {
    "id": "04919992b3",
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "How should I start the course and follow the weekly workflow?",
    "answer": "Start with the [LLM Zoomcamp docs](https://datatalks.club/docs/courses/llm-zoomcamp/), the [general Zoomcamp logistics docs](https://datatalks.club/docs/courses/zoomcamp-logistics/), and the [LLM Zoomcamp GitHub repository](https://github.com/DataTalksClub/llm-zoomcamp).\n\nYou can start whenever you want. The videos and GitHub materials are available, and the deadlines are listed in the [course management platform](https://courses.datatalks.club/llm-zoomcamp-2026/).

In [16]:
tool_result = {
    "role": "user",
    "content": [
        {
            "type": "tool_result",
            "tool_use_id": tool_use.id,
            "content": results_json
        }
    ]
}


In [17]:
response = client.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=1024,
    messages=[
        *messages,                          # original user message
        {"role": "assistant", "content": response.content},  # Claude's tool call response
        tool_result                # your tool result
    ],
    tools=[search_tool]
)

In [18]:
print(response.content[0].text)

Great! Yes, you can join the course! Here's what you need to know:

**You can join anytime**, but there are some important things to keep in mind:

1. **Certificate Eligibility**: If you want to receive a certificate, you need to submit your project while the course is still accepting submissions. Note that certificates are only awarded if you complete the course with a "live" cohort (not in self-paced mode), as you'll also need to peer-review 3 capstone projects after submitting your own.

2. **No Formal Registration Required**: You don't need a confirmation email to start learning. You can:
   - Start learning immediately with the available videos and materials
   - Submit homework through the course platform while the submission form is open
   - You don't need to be on a registered list to participate

3. **Getting Started**: Follow this workflow:
   - Watch the lesson videos
   - Work through the lesson notebooks/code
   - Read the homework instructions on GitHub
   - Submit answe

Now the

5. LLM processes the results
6. LLM gives the answer

In [19]:
usage = response.usage

In [20]:
usage.input_tokens, usage.output_tokens

(1426, 359)

Total Process:

1. Make a call to the LLM <-- (first call)
2. LL decided to invoke search with 'params'
3. Then we invoked the search and got results back
4. Now we'll send the results back to the LLM <-- (another call)
5. LLM processes the results
6. LLM gives the answer

But what if the LLM needs to make another tool call and the process looks like??

1. Make a call to the LLM <-- (first call)
2. LL decided to invoke search with 'params'
3. Then we invoked the search and got results back
4. Now we'll send the results back to the LLM <-- (another call)
5. LLM processes the results
6. LLM decides to make another tool call 
7. Send another API request (call)
8. LLM Processes and gives the answer

^This is what the agentic loop is for

In [36]:
def make_call(tool_use):
    args = tool_use.input  # already a dict, no json.loads needed

    if tool_use.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "role": "user",
        "content": [
            {
                "type": "tool_result",
                "tool_use_id": tool_use.id,
                "content": result_json
            }
        ]
    }

In [37]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
"""

question = "I just discovered the course. Can I join it?"

messages=[
    {"role": "user", "content": question}
]

In [38]:
response = client.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=1024,
    system=instructions,
    messages=messages,
    tools=[search_tool]
)

In [45]:
response.content[0].text

"I'd be happy to help you with information about joining the course! Let me search the FAQ for enrollment and access details."

In [48]:
messages.extend(response.content)

for item in response.content:
    
    if item.type == "tool_use":
        # process function call 
        print(f"Tool use detected: {item.name} with input {item.input}")
        call_output=make_call(item)
        messages.append(call_output)

    elif item.type == "message":
        print('ASSISTANT:')
        print(item.content[0].text)

Tool use detected: search with input {'query': 'join course enrollment registration'}
Tool use detected: search with input {'query': 'can I join when does course start'}


In [41]:
messages

[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 TextBlock(citations=None, text="I'd be happy to help you with information about joining the course! Let me search the FAQ for enrollment and access details.", type='text'),
 ToolUseBlock(id='toolu_01GeZoUto7zML8SM5eGPseiT', caller=DirectCaller(type='direct'), input={'query': 'join course enrollment registration'}, name='search', type='tool_use'),
 ToolUseBlock(id='toolu_01EkkQSRPjzwcv4S3ZiZc2jn', caller=DirectCaller(type='direct'), input={'query': 'can I join when does course start'}, name='search', type='tool_use'),
 {'role': 'user',
  'content': [{'type': 'tool_result',
    'tool_use_id': 'toolu_01GeZoUto7zML8SM5eGPseiT',
    'content': '[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you need to submit your pro

In [51]:
messages=[
    {"role": "user", "content": question}
]


it = 1

while True:

    print(f'iteration #{it}')
    has_tool_use = False

    response = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=1024,
        system=instructions,
        messages=messages,
        tools=[search_tool]
    )

    messages.append({"role": "assistant", "content": response.content})

    has_tool_use = False

    for item in response.content:
        
        if item.type == "tool_use":
            # process function call 
            print(f"Tool use detected: {item.name} with input {item.input}")
            call_output=make_call(item)
            messages.append(call_output)
            has_tool_use = True

        elif item.type == "text":
            print('ASSISTANT:')
            print(item.text)

    it = it + 1

    if has_tool_use == False:
        break


response = client.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=1024,
    system=instructions,
    messages=messages,
    tools=[search_tool]
)

messages.append({"role": "assistant", "content": response.content})

for item in response.content:
    
    if item.type == "tool_use":
        # process function call 
        print(f"Tool use detected: {item.name} with input {item.input}")
        call_output=make_call(item)
        messages.append(call_output)

    elif item.type == "text":
        print('ASSISTANT:')
        print(item.text)

iteration #1
ASSISTANT:
I'll search the FAQ for information about joining the course.
Tool use detected: search with input {'query': 'join course enrollment registration'}
iteration #2
ASSISTANT:
Great news! Based on the course FAQ:

**Yes, you can join the course!** Here are the key points:

1. **You can start anytime** - The course materials (videos, notebooks, and code) are available whenever you want to begin.

2. **No registration required to start learning** - You don't need to formally register to begin working on the course materials and homework. Registration is just used to gauge interest before the course start date.

3. **Certificate eligibility** - If you want to receive a certificate, you'll need to:
   - Submit your project while the course is still accepting submissions
   - Complete the course during a "live" cohort (not in self-paced mode)
   - Participate in peer-reviewing 3 capstone projects after submitting your own

4. **Getting started** - You should:
   - Check 

In [52]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results, and then perform more searches.

At the end, ask if there are other areas that the user wants to explore.
"""

question = "I just discovered the course. Can I join it?"

messages=[
    {"role": "user", "content": question}
]

In [58]:
def agent_loop(question, instructions, model='claude-haiku-4-5-20251001') -> str:
    
    messages = [{"role": "user", "content": question}]
    it = 1
    last_answer = ""


    while True:

        print(f'iteration #{it}')
        has_tool_use = False

        response = client.messages.create(
            model=model,
            max_tokens=1024,
            system=instructions,
            messages=messages,
            tools=[search_tool]
        )

        messages.append({"role": "assistant", "content": response.content})

        has_tool_use = False

        for item in response.content:
            
            if item.type == "tool_use":
                # process function call 
                print(f"Tool use detected: {item.name} with input {item.input}")
                call_output=make_call(item)
                messages.append(call_output)
                has_tool_use = True

            elif item.type == "text":
                print('ASSISTANT:')
                print(item.text)
                last_answer = item.text

        it += 1

        if not has_tool_use:
            break
        
    return last_answer


In [59]:
agent_loop("How do I run ollama locally?", instructions)

iteration #1
ASSISTANT:
I'll search the course FAQ for information about running Ollama locally.
Tool use detected: search with input {'query': 'run ollama locally'}
Tool use detected: search with input {'query': 'ollama setup installation'}
Tool use detected: search with input {'query': 'ollama local installation'}
iteration #2
ASSISTANT:
Great! I found comprehensive information about running Ollama locally. Here's how to do it:

## Installation Steps

**1. Download and Install Ollama**

Visit [https://ollama.com/download](https://ollama.com/download) and choose your operating system:

- **macOS**: Download the `.pkg` and install it
- **Windows**: Download the `.msi` and install it
- **Linux**: Run this command in your terminal:
  ```bash
  curl -fsSL https://ollama.com/install.sh | sh
  ```

**2. Run a Model Locally**

Once installed, open a terminal and run:
```bash
ollama run llama3
```

This command will:
- Download the LLaMA 3 model (~4GB)
- Start the model locally
- Open a chat-

'Great! I found comprehensive information about running Ollama locally. Here\'s how to do it:\n\n## Installation Steps\n\n**1. Download and Install Ollama**\n\nVisit [https://ollama.com/download](https://ollama.com/download) and choose your operating system:\n\n- **macOS**: Download the `.pkg` and install it\n- **Windows**: Download the `.msi` and install it\n- **Linux**: Run this command in your terminal:\n  ```bash\n  curl -fsSL https://ollama.com/install.sh | sh\n  ```\n\n**2. Run a Model Locally**\n\nOnce installed, open a terminal and run:\n```bash\nollama run llama3\n```\n\nThis command will:\n- Download the LLaMA 3 model (~4GB)\n- Start the model locally\n- Open a chat-like interface where you can type questions\n\n**3. Test the Ollama Local Server**\n\nTo verify it\'s running, execute:\n```bash\ncurl http://localhost:11434\n```\n\nYou should receive a response like:\n```json\n{"models": [...]}\n```\n\n## Using Ollama with Python\n\n**1. Install the Python client:**\n```bash\npi